TCGA is The Cancer Genome Atlas, which is a large research program to study the molecular and genetic characteristics of different cancers. The Huggingface subset of TCGA data focused on gene expression which contains 2770 samples, 20531 columns with 5 cancer types. 

Gene expression is a way of measuring how active genes are being used by a cell, a gene in DNA can be transcribed into RNA. The RNA can then be used to produce proteins, so if you measure the RNA produced from genes, you can get an idea of which genes are active and how strong they are being expressed. 
The dataset describes mRNA expression data as measurements generated through mRNA sequencing, with normalized expression values available at the gene. 
A gene-expression profile shows the pattern of gene activity measured for a particuar sample. 

How can ML be useful?

In [ ]:
#import data
from datasets import load_dataset

ds = load_dataset("Fllamber/TCGA-PANCAN-HiSeq-2770x20530")
print(ds)

In [ ]:
#dimensions
ds.shape

In [ ]:
#example of first row
print(ds['train'][0])

The numbers show the measured expression level of the gene in each sample.

In [ ]:
#check the unique cancer types
print(set(ds["train"]["cancer"]))

In [ ]:
import pandas as pd
#convert dataset to pandas datafraem
ds.set_format(type='pandas')
df = ds['train'][:]
df.head()

In [ ]:
#number of samples per cancer type for balanced or imbalanced dataset
df['cancer'].value_counts()

In [ ]:
import matplotlib.pyplot as plt

color = ['green', 'blue', 'yellow', 'red', 'orange']
df['cancer'].value_counts().plot(kind='bar', color=color)
plt.xlabel('Cancer type')
plt.ylabel("number of samples per cancer type")
plt.show()

There is a class imbalance, more BRCA than other cancer types during training

In [ ]:
df.isnull()

In [ ]:
df.isnull().sum()

In [ ]:
#data type
df.info()

In [ ]:
X = df.drop(columns=['cancer']) #features to make predicitions 
y = df['cancer'] #target learn to predict 

In [ ]:
#stratify ensures that the training and testing sets have the same proportion
#of classes as the original dataset, since the dataset is imbalanced
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [ ]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

In [ ]:
print("Training:", X_fit.shape)
print("Validation:", X_val.shape)

In [ ]:
X.describe()

In [ ]:
#standardization
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X_fit)
X_fit_scaled = scaler.transform(X_fit)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#baseline logistic regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report

baseline_logreg = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

baseline_logreg.fit(X_fit_scaled, y_fit)

y_val_pred_baselinelogreg = baseline_logreg.predict(X_val_scaled)

print("Accuracy_score:", accuracy_score(y_val, y_val_pred_baselinelogreg))
print("Baseline logistic regresiion macro-F1:", 
      f1_score(y_val, y_val_pred_baselinelogreg, average="macro"))
print("Classification report:")
print(classification_report(y_val, y_val_pred_baselinelogreg))

In [ ]:
#check best number of principal components
from sklearn.decomposition import PCA

variance_threshold = [0.80, 0.90, 0.95, 0.99]
results_pca = []

for variance in variance_threshold:
    pca_test = PCA(n_components=variance)
    X_fit_pca_test = pca_test.fit_transform(X_fit_scaled)

    X_val_pca_test = pca_test.transform(X_val_scaled)

    model_log = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    )
    model_log.fit(X_fit_pca_test, y_fit)

    y_val_pred_logreg = model_log.predict(X_val_pca_test)

    accuracy = accuracy_score(y_val, y_val_pred_logreg)
    f1_macro = f1_score(y_val, y_val_pred_logreg, average='macro')

    results_pca.append({
        "Variance": variance,
        "Components": pca_test.n_components_,
        "Accuracy": accuracy,
        "Macro f1-score": f1_macro
    })

    print(f"PCA variance: {variance}")
    print(f"Number of components: {pca_test.n_components_}")
    print("Classification report:")
    print(classification_report(
        y_val, 
        y_val_pred_logreg))

results_pca_df = pd.DataFrame(results_pca)
print(results_pca_df.round(3))

In [ ]:
#pca with 0.95 variance
pca = PCA(n_components=0.95)
X_fit_pca = pca.fit_transform(X_fit_scaled)
X_val_pca = pca.transform(X_val_scaled)

print('Variance retained:', pca.explained_variance_ratio_.sum())
print("Original shape:", X_fit_scaled.shape)
print("PCA shape:", X_fit_pca.shape)
print("Number of components:", pca.n_components_)
print("Validation PCA shape:", X_val_pca.shape)

In [ ]:
import numpy as np
print(pca.explained_variance_ratio_)
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
print("Total variance retained:", cumulative_variance[-1])

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(range(1, len(cumulative_variance) +1), cumulative_variance)
plt.axhline(y=0.95, color='red', linestyle='--', label='95% variance')
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA: Cumulative explained variance")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
#t-SNE
from sklearn.manifold import TSNE

pca_tsne = PCA(n_components=50, random_state=42)

X_fit_pca50 = pca_tsne.fit_transform(X_fit_scaled)

tsne = TSNE(n_components=2,
            perplexity=30,
            init="pca",
            learning_rate="auto",
            random_state=42)

X_fit_tsne = tsne.fit_transform(X_fit_pca50)

tsne_df = pd.DataFrame({
    "TSNE_1": X_fit_tsne[:, 0],
    "TSNE_2": X_fit_tsne[:, 1],
    "Cancer": y_fit.to_numpy()
})

plt.figure(figsize=(10,7))

for cancer_type in sorted(tsne_df["Cancer"].unique()):
    subset = tsne_df[tsne_df["Cancer"] == cancer_type]

    plt.scatter(
        subset["TSNE_1"],
        subset['TSNE_2'],
        label=cancer_type,
        alpha=0.7,
        s=25)

plt.title('t-SNE Visualization of Cancer Gene Expression')
plt.xlabel('t-SNE dimension 1')
plt.ylabel('t-SNE dimension 2')
plt.legend(title='Caner type')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
#feature selection
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif

#remove features with no variance
selector_var = VarianceThreshold(threshold=0.0)
remove_constants_X_fit = selector_var.fit_transform(X_fit_scaled)
remove_constants_X_val = selector_var.transform(X_val_scaled)

print("Features", X_fit_scaled.shape[1])
print("Features after removing constants:", remove_constants_X_fit.shape[1])

#genes to keep
k_values = [100, 500, 1000, 2000, 5000, 10000]

selection_results = []

for k in k_values: 
    selector = SelectKBest(score_func=f_classif, k=k)
    X_fit_scaled_selected = selector.fit_transform(remove_constants_X_fit, y_fit)
    X_val_scaled_selected = selector.transform(remove_constants_X_val)

    model_log.fit(X_fit_scaled_selected, y_fit)

    pred_selected = model_log.predict(X_val_scaled_selected)

    accuracy_select = accuracy_score(y_val, pred_selected)
    f1_macro_select = f1_score(y_val, pred_selected, average='macro')
    
    selection_results.append({
            "Selected genes": k,
            "Accuracy": accuracy_select,
            "Macro f1-score": f1_macro_select
        })
    
    print(f"Selected genes: {k}")
    print("Classification report:")
    print(classification_report(y_val, pred_selected))
    
results_selection_df = pd.DataFrame(selection_results)
print("Feature selection results:")
print(results_selection_df.round(3))